# State of Embroidery Pricing 2026
## End-to-End Survey Data Analysis Pipeline

This portfolio piece demonstrates a comprehensive data science workflow, including:
1. **Data Cleaning & Engineering:** Deduplication, string normalization, and categorical mapping of messy survey data.
2. **Feature Engineering & Scoring:** Implementing a scoring engine to evaluate pricing maturity.
3. **Data Segmentation:** Segmenting respondents by business characteristics for granular analysis.
4. **Relational Merging:** Joining disparate datasets using fuzzy/normalized identifiers.
5. **Exploratory Data Analysis (EDA):** Visualizing distributions and key metrics.
6. **Statistical Hypothesis Testing:** Using formal statistics (ANOVA, Chi-Square) to derive business insights.



### Phase 1: Data Cleaning, Preprocessing & Segmentation
We begin by loading the raw datasets. These datasets represent real-world, uncleaned survey exports containing duplicates, partial completions, and inconsistent text inputs.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")

# Define file paths
raw_diag_path = 'nnep_uncleaned_survey_simulation_package/raw/pricing_diagnostic_raw_export.csv'
raw_bench_path = 'nnep_uncleaned_survey_simulation_package/raw/benchmark_survey_raw_export.csv'

# Load data
df_diag = pd.read_csv(raw_diag_path)
df_bench = pd.read_csv(raw_bench_path)

print(f"Raw Diagnostic Shape: {df_diag.shape}")
print(f"Raw Benchmark Shape: {df_bench.shape}")



**Data Cleaning Pipeline**


In [ ]:
def clean_emails(email_series):
    '''Lowercase, strip whitespace, and handle + aliases for emails to create a reliable join key.'''
    return (email_series
            .astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r'\+.*@', '@', regex=True)) # remove + alias

def clean_diagnostic_data(df):
    df_clean = df.copy()
    
    # 1. Filter out partials / test rows
    df_clean = df_clean[
        (df_clean['Finished'].astype(str).str.lower().isin(['true', 'y', 'yes', 'complete'])) &
        (df_clean['Progress'].astype(str) == '100')
    ]
    
    # 2. Deduplicate based on Email (keep the most recent based on End Date)
    df_clean['End Date'] = pd.to_datetime(df_clean['End Date'], errors='coerce')
    df_clean = df_clean.sort_values('End Date').drop_duplicates(subset=['Email'], keep='last')
    
    # 3. Normalize Identifiers
    df_clean['normalized_email'] = clean_emails(df_clean['Email'])
    
    # 4. Scoring Engine (0-2 Scale)
    score_map = {
        'No': 0, 'Not really': 0, 'I usually guess': 0, 'No consistent process': 0,
        'Sometimes': 1, 'For larger jobs only': 1, 'Partially': 1, 'Rough estimate': 1, 'Depends on the job': 1, 'I have a basic spreadsheet': 1,
        'Yes': 2, 'Always': 2, 'Yes, documented': 2, 'Built into our quote process': 2, 'We review and adjust': 2, 'We use a calculator/spreadsheet': 2
    }
    
    q_cols = [c for c in df_clean.columns if c.startswith('Q')]
    for i, col in enumerate(q_cols, 1):
        df_clean[f'q{i}_score'] = df_clean[col].map(score_map).fillna(0)
        
    df_clean['total_diagnostic_score'] = df_clean[[f'q{i}_score' for i in range(1, 11)]].sum(axis=1)
    
    # 5. Segmentation (Pricing Maturity)
    def categorize_maturity(score):
        if score <= 7: return 'Pricing Blind Spot'
        elif score <= 14: return 'Partial Pricing System'
        else: return 'Strong Pricing System'
        
    df_clean['pricing_maturity_segment'] = df_clean['total_diagnostic_score'].apply(categorize_maturity)
    
    # Business Size Segmentation
    size_map = {
        'solo': 'Solo', 'just me': 'Solo', 'one-person shop': 'Solo',
        '2-5': 'Small Team', 'me + 2': 'Small Team', '2-5 employees': 'Small Team', 'small team': 'Small Team',
        'about 8': 'Medium', '6-10 employees': 'Medium', '11-25 employees': 'Medium', 'larger shop': 'Medium',
        '26+': 'Large', '26+ employees': 'Large'
    }
    df_clean['business_size_segment'] = df_clean['Business Size'].str.lower().map(size_map).fillna('Unknown')
    
    return df_clean

df_diag_clean = clean_diagnostic_data(df_diag)
print(f"Cleaned Diagnostic Shape: {df_diag_clean.shape}")



In [ ]:
import re
def extract_numeric(text):
    if pd.isna(text): return np.nan
    matches = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", str(text).replace(',', ''))
    return float(matches[0]) if matches else np.nan

def clean_benchmark_data(df):
    df_clean = df.copy()
    
    # 1. Filter out partials
    df_clean = df_clean[
        (df_clean['Finished'].astype(str).str.lower().isin(['true', 'y', 'yes', 'complete'])) &
        (df_clean['Progress'].astype(str) == '100')
    ]
    
    # 2. Deduplicate
    df_clean['Start Timestamp'] = pd.to_datetime(df_clean['Start Timestamp'], errors='coerce')
    df_clean = df_clean.sort_values('Start Timestamp').drop_duplicates(subset=['Email Address'], keep='last')
    
    # 3. Normalize Identifiers
    df_clean['normalized_email'] = clean_emails(df_clean['Email Address'])
    
    # 4. Metric Extraction
    df_clean['hourly_rate_usd'] = df_clean['Hourly rate used, if any'].apply(extract_numeric)
    
    # Convert confidence to 1-5 numeric scale
    conf_map = {
        '1 - not at all confident': 1, 'not very': 2, '2 - somewhat confident': 2, 
        'Slightly confident': 3, 'somewhat': 3, '3 - moderately confident': 3,
        '4 - very confident': 4, '5 - very confident': 5, 'highly confident': 5
    }
    df_clean['pricing_confidence_score'] = df_clean['How confident are you in your pricing?'].str.lower().map(conf_map).fillna(3)
    
    # Normalize profitability
    df_clean['perceived_profitability'] = df_clean['How would you describe profitability?'].astype(str).str.lower()
    
    return df_clean

df_bench_clean = clean_benchmark_data(df_bench)
print(f"Cleaned Benchmark Shape: {df_bench_clean.shape}")



### Phase 2: Relational Merging
We now merge the two cleaned datasets using the normalized email addresses to analyze the behavior of respondents who took both surveys.


In [ ]:
# Join datasets
df_overlap = pd.merge(df_diag_clean, df_bench_clean, on='normalized_email', how='inner', suffixes=('_diag', '_bench'))
print(f"Overlap Analysis Ready Shape: {df_overlap.shape}")



### Phase 3 & 4: EDA and Statistical Hypothesis Testing

#### Hypothesis 1: Do businesses with a "Strong Pricing System" charge a significantly higher average hourly rate?
**Test:** One-way ANOVA


In [ ]:
# Drop missing hourly rates for this test
h1_data = df_overlap.dropna(subset=['hourly_rate_usd'])

plt.figure(figsize=(10, 6))
sns.boxplot(x='pricing_maturity_segment', y='hourly_rate_usd', data=h1_data, 
            order=['Pricing Blind Spot', 'Partial Pricing System', 'Strong Pricing System'],
            palette="Blues")
plt.title("Hourly Rate by Pricing Maturity Segment", fontsize=14)
plt.ylabel("Hourly Rate (USD)")
plt.xlabel("Pricing Maturity")
plt.show()

# ANOVA Test
groups = [group['hourly_rate_usd'].values for name, group in h1_data.groupby('pricing_maturity_segment')]
f_stat, p_val = stats.f_oneway(*groups)
print(f"ANOVA F-statistic: {f_stat:.2f}")
print(f"p-value: {p_val:.4f}")

if p_val < 0.05:
    print("Conclusion: There is a statistically significant difference in hourly rates across pricing maturity segments.")
else:
    print("Conclusion: No statistically significant difference found.")

